# 04 (LOCAL) — LLM Diagnostic Layer, run on your laptop

Runs the full diagnostic pipeline **locally** — no GPU, no Kaggle, no full dataset download.
It uses your local `models/unet_dropout.pt`, fetches only the JSONs + a **class-stratified sample**
of validation images from Hugging Face (a few MB), and calls **Gemini** (free tier) for each diagnosis.

**Sample size:** default 300, class-stratified — statistically solid, within Gemini's daily free cap,
and enough to report per-class diagnostic quality. Resumable: re-run if you hit a rate limit and it continues.

**Prereqs (local .venv):** `pip install google-generativeai segmentation-models-pytorch albumentations opencv-python huggingface_hub torch`
**API key:** set `GEMINI_API_KEY` as an environment variable before launching Jupyter/VS Code, or set it in cell 1.

## 1. Config + API key

In [1]:
import os, sys
# notebooks/ is the default CWD -> step up to the repo root so 'src' is importable
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
assert os.path.isdir('src'), f'expected repo root, got {os.getcwd()}'
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd(), '| src found:', os.path.isdir('src'))

# --- API key: prefer env var; do NOT hard-code in a shared notebook ---
if not os.environ.get('GEMINI_API_KEY'):
    # uncomment and paste ONLY for a quick local test, then remove before committing:
    # os.environ['GEMINI_API_KEY'] = 'your-key-here'
    pass
assert os.environ.get('GEMINI_API_KEY'), 'set GEMINI_API_KEY env var (aistudio.google.com)'

PROVIDER   = 'gemini'
CKPT       = 'models/unet_dropout.pt'
N_SAMPLE   = 40          # class-stratified sample size
MC_PASSES  = 20           # MC-dropout passes for confidence (fewer = faster on CPU)
OUT_DIR    = 'results/llm'
os.makedirs(OUT_DIR, exist_ok=True)
print('key set:', bool(os.environ.get('GEMINI_API_KEY')), '| ckpt exists:', os.path.exists(CKPT))

cwd: c:\Users\HP PC\Projects\SteelDefectX | src found: True
key set: True | ckpt exists: True


## 2. Fetch ONLY the JSONs + a class-stratified image sample (a few MB)

In [2]:
import json, random
from collections import defaultdict
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import EntryNotFoundError
REPO = 'Zhaosxian/SteelDefectX'

def get_json(fn):
    p = hf_hub_download(repo_id=REPO, filename=fn, repo_type='dataset')
    return json.load(open(p, encoding='utf-8'))

t1 = get_json('class_descriptions.json')
val_text = get_json('val-text.json')
print('classes:', len(t1), '| val entries:', len(val_text))

# class-stratified sample: ~proportional per class, all classes represented
random.seed(0)
by_cls = defaultdict(list)
for e in val_text:
    by_cls[e['class_name']].append(e)
per_class = max(1, N_SAMPLE // len(by_cls))
sample = []
for c, items in by_cls.items():
    random.shuffle(items)
    sample += items[:per_class]
random.shuffle(sample)
sample = sample[:N_SAMPLE]
print(f'stratified sample: {len(sample)} images across {len(by_cls)} classes')

def fetch_val_image(image_name):
    return hf_hub_download(repo_id=REPO, filename=f'val/{image_name}', repo_type='dataset')

classes: 25 | val entries: 2324
stratified sample: 25 images across 25 classes


## 3. Load the local dropout-U-Net

In [3]:
import numpy as np, cv2, torch
from src.segmentation.model import build_model, enable_mc_dropout
from src.llm.attributes import extract_attributes
from src.llm.diagnose import diagnose
from src.llm.evaluate_llm import evaluate_batch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model('unet', 'resnet34', None, dropout=0.2)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.to(device).eval()
print('model loaded on', device)

MEAN = np.array([0.485,0.456,0.406])[None,:,None,None]
STD  = np.array([0.229,0.224,0.225])[None,:,None,None]

def predict_with_conf(img_path, mc=MC_PASSES):
    g = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE); g = cv2.resize(g, (256,256))
    x = np.stack([g,g,g],0)[None].astype(np.float32)/255.0
    x = torch.tensor((x-MEAN)/STD, dtype=torch.float32).to(device)
    enable_mc_dropout(model)
    with torch.no_grad():
        ps = torch.stack([torch.sigmoid(model(x)) for _ in range(mc)],0)
    prob = ps.mean(0)[0,0].cpu().numpy(); unc = ps.std(0)[0,0].cpu().numpy()
    conf = float(prob[prob>0.5].mean()) if (prob>0.5).any() else float(1-prob.mean())
    return g, prob, unc, conf

model loaded on cpu


## 4. Run the pipeline over the sample (RESUMABLE)

Saves each diagnosis as it goes. If you hit a Gemini rate limit, just re-run this cell — it skips
images already done and continues.

In [6]:
import time
PROG = f'{OUT_DIR}/diagnoses.jsonl'
done = set()
if os.path.exists(PROG):
    for line in open(PROG):
        try: done.add(json.loads(line)['image_name'])
        except Exception: pass
print('already done:', len(done))

fout = open(PROG, 'a')
errors = 0
for i, e in enumerate(sample):
    nm, cl = e['image_name'], e['class_name']
    if nm in done:
        continue
    try:
        g, prob, unc, conf = predict_with_conf(fetch_val_image(nm))
        attrs = extract_attributes(g, prob)
        diag = diagnose(cl, t1[cl], attrs, confidence=conf, uncertainty=float(unc.mean()), provider=PROVIDER, model='gemini-3.6-flash')
        rec = {'image_name': nm, 'class_name': cl, 'confidence': round(conf,3),
               'attributes': {k:v for k,v in attrs.items() if k!='_raw'}, 'diag': diag}
        fout.write(json.dumps(rec)+'\n'); fout.flush()
        if (i+1) % 20 == 0: print(f'{i+1}/{len(sample)} done')
    except Exception as ex:
        errors += 1
        msg = str(ex)
        if 'rate' in msg.lower() or 'quota' in msg.lower() or '429' in msg:
            print(f'rate limit at {i} — sleeping 60s then continuing...')
            time.sleep(60)
            # retry this same image once after the wait
            try:
                g, prob, unc, conf = predict_with_conf(fetch_val_image(nm))
                attrs = extract_attributes(g, prob)
                diag = diagnose(cl, t1[cl], attrs, confidence=conf, uncertainty=float(unc.mean()), provider=PROVIDER)                
                rec = {'image_name': nm, 'class_name': cl, 'confidence': round(conf,3),
                    'attributes': {k:v for k,v in attrs.items() if k!='_raw'}, 'diag': diag}
                fout.write(json.dumps(rec)+'\n'); fout.flush()
            except Exception:
                pass
            continue
        print('skip', nm, msg[:120])
        time.sleep(1)
fout.close()
print('errors:', errors)

already done: 0


val/crease_04.jpg: reconstructing file:   0%|          |  0.00B / 4.81kB            

val/crease_04.jpg: downloading bytes:           |  0.00B            

c:\Users\HP PC\Projects\SteelDefectX\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP PC\.cache\huggingface\hub\datasets--Zhaosxian--SteelDefectX. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


val/wl_443.jpg: reconstructing file:   0%|          |  0.00B / 6.63kB            

...

val/pa_138.jpg: downloading bytes:           |  0.00B            

rate limit at 22 — sleeping 60s then continuing...


val/ds_02.jpg: reconstructing file:   0%|          |  0.00B / 6.90kB            

val/ds_02.jpg: downloading bytes:           |  0.00B            

rate limit at 23 — sleeping 60s then continuing...
rate limit at 24 — sleeping 60s then continuing...
errors: 4


## 5. Aggregate metrics: overall + per-class

In [7]:
from src.llm.evaluate_llm import check_structural_validity, check_grounding, check_faithfulness
recs = [json.loads(l) for l in open(PROG)]
print('total diagnoses:', len(recs))

batch = [{'diag': r['diag'], 't1_cause': t1[r['class_name']], 'attributes': r['attributes']} for r in recs]
overall = evaluate_batch(batch)
print('\nOVERALL:', json.dumps(overall, indent=2))

# per-class
from collections import defaultdict
pc = defaultdict(lambda: {'n':0,'valid':0,'ground':0,'faith':0})
for r in recs:
    c = r['class_name']; d = r['diag']; a = r['attributes']
    pc[c]['n'] += 1
    pc[c]['valid']  += int(check_structural_validity(d))
    pc[c]['ground'] += int(check_grounding(d, t1[c]))
    pc[c]['faith']  += int(check_faithfulness(d, a))
print('\nPER-CLASS (validity / grounding / faithfulness %):')
for c in sorted(pc):
    s = pc[c]; n = s['n']
    print(f"  {c:32s} n={n:3d}  val={100*s['valid']/n:5.1f}  grd={100*s['ground']/n:5.1f}  fth={100*s['faith']/n:5.1f}")

json.dump({'overall': overall, 'per_class': {c: dict(v) for c,v in pc.items()}},
          open(f'{OUT_DIR}/eval_metrics.json','w'), indent=2)
print('\nsaved -> results/llm/eval_metrics.json')

total diagnoses: 21

OVERALL: {
  "n": 21,
  "structural_validity_pct": 100.0,
  "grounding_rate_pct": 100.0,
  "faithfulness_pct": 100.0
}

PER-CLASS (validity / grounding / faithfulness %):
  Bright scratch                   n=  1  val=100.0  grd=100.0  fth=100.0
  Crazing                          n=  1  val=100.0  grd=100.0  fth=100.0
  Crease                           n=  1  val=100.0  grd=100.0  fth=100.0
  Crescent gap                     n=  1  val=100.0  grd=100.0  fth=100.0
  Finishing roll printing          n=  1  val=100.0  grd=100.0  fth=100.0
  Inclusion                        n=  1  val=100.0  grd=100.0  fth=100.0
  Iron scale compression           n=  1  val=100.0  grd=100.0  fth=100.0
  Iron sheet ash                   n=  1  val=100.0  grd=100.0  fth=100.0
  Oil spot                         n=  1  val=100.0  grd=100.0  fth=100.0
  Oxide scale of plate system      n=  1  val=100.0  grd=100.0  fth=100.0
  Oxide scale of temperature system n=  1  val=100.0  grd=100.0  fth

## 6. Show a few example diagnostic notes (for the report / demo)

In [8]:
for r in recs[:4]:
    d = r['diag']
    print(f"=== {r['image_name']} [{r['class_name']}]  conf={r['confidence']} ===")
    print('  attributes:', {k:r['attributes'][k] for k in ('Shape','Scale','Polarity','Saliency') if k in r['attributes']})
    print('  cause   :', d.get('likely_cause'))
    print('  severity:', d.get('severity'))
    print('  action  :', d.get('recommended_action'))
    print('  summary :', d.get('summary'))
    print()

=== Ops_24.jpg [Oxide scale of plate system]  conf=0.963 ===
  attributes: {'Shape': 'linear', 'Scale': 'medium', 'Polarity': 'dark', 'Saliency': 'medium'}
  cause   : Embedded oxide particles resulting from work roller damage during high-speed hot rolling.
  severity: moderate
  action  : Inspect hot rolling mill work rollers for surface damage and hold the plate for localized surface re-examination.
  summary : An isolated vertical dark linear oxide scale mark is present on the lower-right and center regions of the plate, consistent with damaged hot rolling rolls.

=== crease_04.jpg [Crease]  conf=0.97 ===
  attributes: {'Shape': 'linear', 'Scale': 'small', 'Polarity': 'dark', 'Saliency': 'medium'}
  cause   : Localized yielding of the strip material during uncoiling.
  severity: low
  action  : Inspect the uncoiling station equipment, specifically checking strip tension control and leveller entry rolls.
  summary : Three small, clustered horizontal crease defects were detected near 

In [5]:
# debug: see the raw Gemini response for one case
import json
from src.llm.diagnose import build_user_prompt, _call_llm, SYSTEM_PROMPT
from src.llm.attributes import extract_attributes

e = sample[0]
nm, cl = e['image_name'], e['class_name']
g, prob, unc, conf = predict_with_conf(fetch_val_image(nm))
attrs = extract_attributes(g, prob)
user = build_user_prompt(cl, t1[cl], attrs, conf, float(unc.mean()))
raw = _call_llm(SYSTEM_PROMPT, user, provider='gemini', model='gemini-3.6-flash')
print("RAW GEMINI RESPONSE:")
print(repr(raw))

RAW GEMINI RESPONSE:
'{\n  "likely_cause": "Embedded oxide particles pressed into the plate surface due to work roller damage during high-speed hot rolling.",\n  "severity": "moderate",\n  "recommended_action": "Inspect the hot rolling mill work rollers for surface degradation and audit the affected plate segment.",\n  "summary": "A single dark vertical linear oxide scale defect was detected, likely caused by damaged hot rolling work rollers."\n}'
